# MSA Colab: Pre-computed MSA Search, AFDB Templates & PDB Hits

## Overview

Standalone Colab notebook for pre-computing Multiple Sequence Alignments (MSAs) using the
ColabFold MMseqs2 API, plus optional AFDB structural template extraction and PDB best-hit
downloading. Outputs `.a3m` files that are directly consumed by folding tool notebooks
(IntelliFold, Boltz-2, Protenix, Chai-1, OpenFold3).

**No GPU required** -- this notebook runs on CPU. All computation is MSA search (network I/O),
template download, and file processing (CPU-only).

## Why

1. **Speed**: MSA search takes 30-120s per protein. Pre-compute once, reuse across all tools and re-runs.
2. **AFDB Templates**: Automatically extract high-quality AlphaFold DB structural templates for each protein chain.
3. **PDB Hits**: Download the best experimental PDB structure (>30% identity) for each protein chain.
4. **Consistency**: All 5 folding tools get identical MSAs, templates, and structural references.

## Workflow

1. Upload your CSV (same unified format as all folding notebooks)
2. Configure MSA search settings and template/PDB extraction
3. Run: searches MSAs for each unique protein, extracts AFDB templates and PDB hits (optional), writes output
4. Download results from Google Drive

## CSV Input Format

Same columns as all folding notebooks. Only `protein` type sequences are used for MSA search
(DNA, RNA, ligands are ignored).

```
jobname, seq1_name, seq1_type, seq1, seq1_copies, seq1_mods, seq2_name, seq2_type, seq2, ...
```

## Output Structure

```
MSA_Results/
  individual/                           Per-protein full MSAs (reusable across jobs)
    proteinA.a3m
    proteinB.a3m
  paired/                               Per-job query-only MSAs (for pairedMsaPath)
    job1_proteinA_paired.a3m            Query sequence only
    job1_proteinB_paired.a3m            Query sequence only
  templates/                            Per-chain structural data (optional)
    proteinA/
      AFDB/
        P12345.cif                      AFDB predicted structure (original name)
        proteinA_templates.a3m          Template alignment file
      PDB/
        1ABC.cif                        Best PDB hit (original name)
        2DEF.cif                        Second hit (if available)
    proteinB/
      AFDB/
        Q67890.cif
        proteinB_templates.a3m
      PDB/
        3GHI.cif
  metadata.json                         Search stats, template/PDB decisions, timing
```

### paired/ directory

The `paired/` files contain **query sequence only** (2 lines: header + sequence). This matches
the local pipeline behavior where pairing is intentionally removed. Folding tools use:
- `pairedMsaPath` -> `paired/` file (query-only, structural context)
- `unpairedMsaPath` -> `individual/` file (full MSA, evolutionary context)

## AFDB Templates

When `extract_templates` is enabled, the notebook:

1. Parses UniProt accessions from each protein's MSA (ColabFold headers contain UniRef100 IDs)
2. Downloads candidate AFDB structures (cached for reuse)
3. Scores by pLDDT in the aligned region
4. Selects the best template using tiered selection:
   - **Tier 1**: 100% identity + pLDDT >= 85 (instant accept)
   - **Tier 2**: >95% identity, highest pLDDT (>= 70)
   - **Tier 3**: Best overall pLDDT among all qualifying hits (>= 30% identity)
5. Writes Protenix-format template `.a3m` files to `templates/AFDB/`

**All-or-nothing**: Templates are used for a job only if ALL protein chains have a template.
Per-job decisions are recorded in `metadata.json`.

## PDB Hits

When `extract_templates` is enabled, the notebook also searches the RCSB PDB for the best
experimental structure matching each protein sequence (>30% identity):

1. Queries RCSB PDB Search API (MMseqs2 backend) for each protein sequence
2. Downloads the best-scoring hit structure in mmCIF format
3. Fetches metadata (resolution, title, release date) from RCSB Data API
4. Saves structures to `templates/PDB/`

PDB hits complement AFDB templates: AFDB provides predicted structures for any protein with a
UniProt entry, while PDB provides experimentally determined structures (X-ray, cryo-EM, NMR)
that may be higher quality but are only available for proteins that have been experimentally solved.

## Settings Reference

| Setting | Options | Effect |
|---------|---------|--------|
| `msa_server_url` | URL (default: `https://api.colabfold.com`) | ColabFold API endpoint |
| `msa_database` | `UniRef+EnvDB` / `UniRef only` | Broader (recommended) vs faster search |
| `pair_mode` | `unpaired_paired` / `paired` / `unpaired` | What ColabFold server returns |
| `max_msa` | `auto` / `512:1024` / etc. | Truncate MSA depth to save memory in folding |
| `rate_limit_delay` | 1-30s (default 5) | Delay between API requests (rate limiting) |
| `extract_templates` | True / False | Download AFDB templates and best PDB hits |

## Using Results with Folding Notebooks

1. Run this notebook -- results upload to Google Drive automatically
2. In any folding notebook's **Cell 4** (Upload CSV), set `msa_folder` to the output folder
   name (e.g., `MSA_Results`). The notebook auto-downloads and unzips from Google Drive.
3. The folding notebook matches files by name: `{jobname}_{seq_name}_paired.a3m`
   - Name matching is case-insensitive and hyphen/underscore-tolerant
   - All-or-nothing per job: either all protein chains have MSAs, or the whole job uses online search

**Example**: If your CSV has `jobname=Rx-ADP, seq1_name=Rx, seq2_name=ADP`,
the paired MSA files will be `Rx-ADP_Rx_paired.a3m` (protein chains only -- ligands are skipped).

## Per-Tool Consumption

| Tool | Mechanism | Format | Mixed mode? |
|------|-----------|--------|-------------|
| IntelliFold | `msa:` field in YAML per chain | `.a3m` (rows pre-ordered) | Yes |
| Boltz-2 | `msa:` field / `.csv` format | `.csv` with explicit `key` column | Yes |
| Protenix | `pairedMsaPath` + `unpairedMsaPath` in JSON | `.a3m` per chain | Yes |
| Chai-1 | `--msa-directory` CLI flag | `.aligned.pqt` with `pairing_key` | No (all-or-nothing) |
| OpenFold3 | `paired_msa_file_paths` in JSON | `.a3m` per chain | Yes |

## Citation

Mirdita M, Schutze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M.
ColabFold: Making protein folding accessible to all. *Nature Methods* (2022).

## License

Private / not for public release.


In [ ]:
#@title Cell 1: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess, sys, os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
enable_gdrive = True  #@param {type:"boolean"}
#@markdown ---
output_folder_name = "MSA_Results"  #@param {type:"string"}
#@markdown Name for the output folder (local and GDrive). Use descriptive names to
#@markdown distinguish runs (e.g., `MSA_batch1`, `MSA_hostpathogen_run2`).

gdrive_parent_folder = ""  #@param {type:"string"}
#@markdown Google Drive parent folder ID. Leave empty for My Drive root.
#@markdown Find folder ID from URL: `drive.google.com/drive/folders/{FOLDER_ID}`
#@markdown ---
#@markdown **Upload your CSV file below.** Only protein sequences will be used for MSA search.

print("\u25b6 Upload your CSV file:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded. Please upload a CSV file.")

csv_filename = list(uploaded.keys())[0]
csv_path = os.path.join("/content", csv_filename)

# If file was uploaded to a different location, copy it
if not os.path.exists(csv_path):
    for name, content in uploaded.items():
        with open(os.path.join("/content", name), "wb") as f:
            f.write(content)
        csv_path = os.path.join("/content", name)
        break

print(f"\n\u2713 Uploaded: {csv_filename}")

# Sanitize output folder name
import re as _re
output_folder_name = _re.sub(r'[^\w\-.]', '_', output_folder_name.strip()) or "MSA_Results"

# Google Drive setup
drive = None
folder_id = None

if enable_gdrive:
    print(f"\n\u25b6 Connecting to Google Drive...")
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        def get_or_create_folder(drv, folder_name, parent_id="root"):
            file_list = drv.ListFile({
                "q": f"'{{parent_id}}' in parents and title='{{folder_name}}' "
                     f"and mimeType='application/vnd.google-apps.folder' and trashed=false"
            }).GetList()
            if file_list:
                return file_list[0]["id"]
            folder = drv.CreateFile({
                "title": folder_name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [{"id": parent_id}]
            })
            folder.Upload()
            return folder["id"]

        parent_id = gdrive_parent_folder.strip() if gdrive_parent_folder.strip() else "root"
        folder_id = get_or_create_folder(drive, output_folder_name, parent_id=parent_id)
        parent_label = f"parent={parent_id}" if parent_id != "root" else "My Drive root"
        print(f"  \u2713 Google Drive connected. Folder: {output_folder_name} ({parent_label}, ID: {folder_id})")

    except Exception as e:
        print(f"  \u2717 Google Drive connection failed: {e}")
        print(f"    Results will be available for local download only.")
        drive = None
        folder_id = None

# Store settings (jobs added in Cell 3 after CSV processing)
global_settings = {
    "jobs": [],
    "unique_proteins": {},
    "csv_path": csv_path,
    "csv_filename": csv_filename,
    "drive": drive,
    "folder_id": folder_id,
    "gdrive_enabled": enable_gdrive and drive is not None,
    "config": {},
    "output_folder_name": output_folder_name,
}

print("\n" + "\u2501" * 60)
print(f"\u2713 CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + search.")
print("\u2501" * 60)

In [ ]:
#@title Cell 2: Install Dependencies
#@markdown Installs `biopython` and `pydrive2`. No kernel restart needed.
%%time

import subprocess, sys, os

print("\u2501" * 60)
print("MSA Colab \u2014 Installing dependencies")
print("\u2501" * 60)

# ── Install packages ──────────────────────────────────────────
packages = [
    "biopython==1.85",
    "pydrive2==1.21.3",
]

print(f"\n\u25b6 Installing {len(packages)} packages...")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    status = "\u2713" if result.returncode == 0 else "\u2717 FAILED"
    print(f"  {status}  {pkg}")
    if result.returncode != 0:
        print(f"    stderr: {result.stderr.strip()}")

# ── Verify imports ────────────────────────────────────────────
print("\n\u25b6 Verifying imports...")
verify_ok = True
for mod_name, import_name in [("biopython", "Bio"), ("pydrive2", "pydrive2")]:
    try:
        __import__(import_name)
        print(f"  \u2713  {mod_name}")
    except ImportError as e:
        print(f"  \u2717  {mod_name}: {e}")
        verify_ok = False

# Also verify stdlib modules we need
for mod_name in ["requests", "pandas", "hashlib", "tarfile", "io", "json", "csv"]:
    try:
        __import__(mod_name)
        print(f"  \u2713  {mod_name}")
    except ImportError as e:
        print(f"  \u2717  {mod_name}: {e}")
        verify_ok = False

if verify_ok:
    print("\n" + "\u2501" * 60)
    print("\u2713 All dependencies verified. Ready to proceed.")
    print("\u2501" * 60)
else:
    print("\n" + "\u2501" * 60)
    print("\u2717 Some imports failed. Check errors above.")
    print("\u2501" * 60)

In [ ]:
#@title Cell 3: MSA Processor Setup
#@markdown Defines the `MSAProcessor` class with CSV parsing, ColabFold API client,
#@markdown A3M parsing, and AFDB template extraction.

import os, sys, csv, json, hashlib, time, re, io, tarfile, gzip
import requests
import pandas as pd
from collections import defaultdict, OrderedDict


class MSAProcessor:
    # Parses unified CSV, searches MSAs via ColabFold API, and extracts AFDB templates.

    PROTEIN_CHARS = set("ACDEFGHIKLMNPQRSTVWY")
    MAX_SEQS_OPTIONS = {"auto": 0, "512:1024": (512, 1024), "256:512": (256, 512),
                        "128:256": (128, 256), "64:128": (64, 128)}

    AFDB_URL_TEMPLATE = "https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v{version}.cif"
    AFDB_VERSIONS = [6, 4]

    def __init__(self):
        self.jobs = []
        self.unique_proteins = OrderedDict()
        self.msa_results = {}

    # == CSV Parsing ===================================================

    def process_csv(self, csv_path):
        # Parse unified CSV, extract protein sequences, return jobs list and summary DataFrame.
        jobs = []
        with open(csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row_idx, row in enumerate(reader):
                jobname = row.get("jobname", "").strip()
                if not jobname:
                    jobname = f"job_{row_idx + 1}"
                chains = []
                for i in range(1, 11):
                    prefix = f"seq{i}"
                    name = row.get(f"{prefix}_name", "").strip()
                    seq_type = row.get(f"{prefix}_type", "").strip().lower()
                    seq = row.get(f"{prefix}", "").strip()
                    copies = row.get(f"{prefix}_copies", "1").strip()
                    mods = row.get(f"{prefix}_mods", "").strip()
                    if not seq:
                        continue
                    try:
                        copies = max(1, int(copies))
                    except (ValueError, TypeError):
                        copies = 1
                    chains.append({
                        "name": name, "type": seq_type, "sequence": seq,
                        "copies": copies, "mods": mods
                    })
                if chains:
                    jobs.append({"jobname": jobname, "chains": chains})

        self.jobs = jobs

        # Build summary DataFrame
        summary_rows = []
        for job in jobs:
            protein_count = sum(1 for c in job["chains"] if c["type"] == "protein")
            other_count = len(job["chains"]) - protein_count
            total_residues = sum(len(c["sequence"]) for c in job["chains"]
                                if c["type"] == "protein")
            summary_rows.append({
                "Job": job["jobname"],
                "Proteins": protein_count,
                "Other chains": other_count,
                "Total residues": total_residues,
            })
        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

    def get_unique_proteins(self, jobs):
        # Deduplicate (name, sequence) pairs across all jobs. Returns OrderedDict: filename -> sequence.
        seen_seqs = {}
        seen_names = {}
        unique = OrderedDict()

        for job in jobs:
            for chain in job["chains"]:
                if chain["type"] != "protein":
                    continue
                seq = chain["sequence"].upper().strip()
                name = chain["name"].strip()

                # Check if this exact sequence was already seen
                seq_hash = hashlib.sha256(seq.encode()).hexdigest()
                if seq in seen_seqs:
                    chain["_msa_filename"] = seen_seqs[seq]
                    continue

                # Determine filename
                if not name:
                    filename = seq_hash
                elif name in seen_names and seen_names[name] != seq:
                    filename = f"{name}_{seq_hash[:8]}"
                else:
                    filename = name

                seen_names[name] = seq if name else None
                seen_seqs[seq] = filename
                chain["_msa_filename"] = filename
                unique[filename] = seq

        self.unique_proteins = unique
        return unique

    # == ColabFold API Client ==========================================

    def search_msa(self, sequence, server_url="https://api.colabfold.com",
                   database="uniref_envdb", pair_mode="unpaired_paired"):
        # POST to ColabFold /ticket/msa, poll for results, download and extract .a3m.
        # Returns raw a3m content string on success, None on failure.

        submit_url = f"{server_url}/ticket/msa"
        payload = {
            "q": f">query\n{sequence}\n",
            "mode": pair_mode,
            "database": database,
        }

        # Submit job
        for attempt in range(3):
            try:
                resp = requests.post(submit_url, data=payload, timeout=60)
                if resp.status_code == 200:
                    break
                if resp.status_code == 429:
                    wait = int(resp.headers.get("Retry-After", 30))
                    print(f"    Rate limited. Waiting {wait}s...")
                    time.sleep(wait)
                    continue
                print(f"    Submit error: HTTP {resp.status_code}")
                if attempt == 2:
                    return None
                time.sleep(5)
            except requests.RequestException as e:
                print(f"    Request error: {e}")
                if attempt == 2:
                    return None
                time.sleep(5)

        try:
            ticket_data = resp.json()
        except (ValueError, KeyError):
            print(f"    Invalid response from server")
            return None

        ticket_id = ticket_data.get("id")
        if not ticket_id:
            print(f"    No ticket ID in response")
            return None

        # Poll for results
        poll_url = f"{server_url}/ticket/{ticket_id}"
        max_polls = 360  # up to 30 min
        for poll_count in range(max_polls):
            time.sleep(5)
            try:
                poll_resp = requests.get(poll_url, timeout=30)
            except requests.RequestException:
                continue

            if poll_resp.status_code == 200:
                poll_data = poll_resp.json()
                status = poll_data.get("status")
                if status == "COMPLETE":
                    break
                elif status == "ERROR":
                    print(f"    Server returned error for ticket {ticket_id}")
                    return None
            elif poll_resp.status_code == 404:
                print(f"    Ticket {ticket_id} not found")
                return None

            if poll_count > 0 and poll_count % 12 == 0:
                mins = poll_count * 5 / 60
                print(f"    Still waiting... ({mins:.0f} min)")
        else:
            print(f"    Timeout waiting for MSA results")
            return None

        # Download results
        download_url = f"{server_url}/result/download/{ticket_id}"
        try:
            dl_resp = requests.get(download_url, timeout=120)
            if dl_resp.status_code != 200:
                print(f"    Download failed: HTTP {dl_resp.status_code}")
                return None
        except requests.RequestException as e:
            print(f"    Download error: {e}")
            return None

        # Extract a3m from tar.gz
        try:
            tar_bytes = io.BytesIO(dl_resp.content)
            with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
                for member in tar.getmembers():
                    if member.name.endswith(".a3m"):
                        f = tar.extractfile(member)
                        if f:
                            content = f.read().decode("utf-8", errors="replace")
                            return content
            print(f"    No .a3m file found in download archive")
            return None
        except (tarfile.TarError, gzip.BadGzipFile) as e:
            try:
                content = dl_resp.content.decode("utf-8", errors="replace")
                if content.startswith(">"):
                    return content
            except Exception:
                pass
            print(f"    Failed to extract results: {e}")
            return None

    # == A3M Parsing ===================================================

    def parse_a3m_headers(self, a3m_content):
        # Parse a3m content into list of entries: [{"header": str, "sequence": str, "taxid": int|None, "species": str}].
        entries = []
        lines = a3m_content.strip().split("\n")
        current_header = None
        current_seq_parts = []

        for line in lines:
            line = line.rstrip()
            if not line or line.startswith("#"):
                continue
            if line.startswith(">"):
                if current_header is not None:
                    seq = "".join(current_seq_parts)
                    taxid = self._extract_taxid(current_header)
                    species = self._extract_species(current_header)
                    entries.append({
                        "header": current_header, "sequence": seq,
                        "taxid": taxid, "species": species
                    })
                current_header = line
                current_seq_parts = []
            else:
                current_seq_parts.append(line)

        # Last entry
        if current_header is not None:
            seq = "".join(current_seq_parts)
            taxid = self._extract_taxid(current_header)
            species = self._extract_species(current_header)
            entries.append({
                "header": current_header, "sequence": seq,
                "taxid": taxid, "species": species
            })

        return entries

    def _extract_taxid(self, header):
        # Extract OX=NNNNN from header. Returns int or None.
        match = re.search(r"OX=(\d+)", header)
        return int(match.group(1)) if match else None

    def _extract_species(self, header):
        # Extract OS=... from header. Returns string or empty.
        match = re.search(r"OS=([^=]+?)(?:\s+\w+=|$)", header)
        return match.group(1).strip() if match else ""

    # == AFDB Template Extraction ======================================

    def _extract_uniprot_id(self, header):
        """Extract UniProt accession from ColabFold a3m header.

        ColabFold headers use tab-separated fields:
            >UniRef100_Q00363\t100\t1.00\t4.090E-22\t0\t73\t74\t38\t111\t135
        Also handles: sp|P12345|..., tr|A0A123|..., bare accession.
        """
        # Strip > prefix
        h = header.lstrip(">").split("\t")[0].strip()

        # UniRef100_ACCESSION
        match = re.match(r"^UniRef\d+_([A-Z0-9]+)", h)
        if match:
            return match.group(1)

        # sp|ACCESSION|... or tr|ACCESSION|...
        match = re.match(r"^[a-z]{2}\|([A-Z0-9]+)\|", h)
        if match:
            return match.group(1)

        # Bare accession
        match = re.match(r"^([A-Z][A-Z0-9]{5,9})\b", h)
        if match:
            return match.group(1)

        return None

    def _parse_colabfold_hit(self, header):
        """Parse ColabFold tab-delimited header fields.

        Format: >ID\tscore\tidentity\tevalue\tqstart\tqend\tqlen\ttstart\ttend\ttlen
        Returns dict with parsed fields or None if parsing fails.
        """
        parts = header.lstrip(">").split("\t")
        if len(parts) < 10:
            return None
        try:
            return {
                "id": parts[0].strip(),
                "score": float(parts[1]),
                "identity": float(parts[2]),
                "evalue": float(parts[3]),
                "qstart": int(parts[4]),
                "qend": int(parts[5]),
                "qlen": int(parts[6]),
                "tstart": int(parts[7]),
                "tend": int(parts[8]),
                "tlen": int(parts[9]),
            }
        except (ValueError, IndexError):
            return None

    def download_afdb_structure(self, uniprot_id, cache_dir):
        """Download mmCIF from AFDB, cache at cache_dir/{uniprot_id}.cif.

        Tries AFDB v6 first, falls back to v4.
        Returns cached path or None on 404/error.
        """
        import urllib.request, urllib.error
        os.makedirs(cache_dir, exist_ok=True)
        cached = os.path.join(cache_dir, f"{uniprot_id}.cif")

        if os.path.exists(cached):
            return cached

        for version in self.AFDB_VERSIONS:
            url = self.AFDB_URL_TEMPLATE.format(uniprot_id=uniprot_id, version=version)
            try:
                urllib.request.urlretrieve(url, cached)
                return cached
            except urllib.error.HTTPError as e:
                if e.code == 404:
                    continue
                return None
            except Exception:
                return None

        return None

    def parse_plddt_for_region(self, cif_path, start, end):
        """Parse mean pLDDT for residue range [start, end] (1-based) from AFDB mmCIF.

        AFDB stores per-residue pLDDT in the B-factor column. We read CA atoms only.
        Returns mean pLDDT or 0.0 if parsing fails.
        """
        try:
            plddt_values = []
            in_atom_site = False
            col_indices = {}
            col_count = 0

            with open(cif_path) as f:
                for line in f:
                    line = line.rstrip()

                    if line.startswith("_atom_site."):
                        in_atom_site = True
                        col_indices[line.strip()] = col_count
                        col_count += 1
                        continue

                    if in_atom_site and line.startswith("_"):
                        in_atom_site = False
                        col_indices.clear()
                        col_count = 0
                        continue

                    if not col_indices:
                        continue

                    if line.startswith("ATOM") or line.startswith("HETATM"):
                        fields = line.split()
                        if len(fields) < col_count:
                            continue

                        atom_idx = col_indices.get("_atom_site.label_atom_id")
                        seq_idx = col_indices.get("_atom_site.label_seq_id")
                        bfac_idx = col_indices.get("_atom_site.B_iso_or_equiv")

                        if atom_idx is None or seq_idx is None or bfac_idx is None:
                            continue

                        if fields[atom_idx] != "CA":
                            continue

                        try:
                            res_num = int(fields[seq_idx])
                            bfactor = float(fields[bfac_idx])
                        except (ValueError, IndexError):
                            continue

                        if start <= res_num <= end:
                            plddt_values.append(bfactor)

            if not plddt_values:
                return 0.0
            return sum(plddt_values) / len(plddt_values)

        except Exception:
            return 0.0

    def extract_best_template(self, a3m_content, query_seq, cache_dir,
                              max_candidates=5, min_identity=0.30,
                              plddt_threshold=70.0, perfect_plddt=85.0):
        """Extract the single best AFDB template from a ColabFold .a3m.

        Parses UniProt hits from headers, downloads AFDB structures,
        scores by pLDDT, and returns the best using tiered selection.

        Returns dict with template info or None if no suitable template found.
        """
        entries = self.parse_a3m_headers(a3m_content)

        # Collect candidates from MSA hits (skip query at index 0)
        candidates = []
        seen_ids = set()
        for entry in entries[1:]:
            hit = self._parse_colabfold_hit(entry["header"])
            if hit is None:
                continue
            if hit["identity"] < min_identity:
                continue
            uniprot_id = self._extract_uniprot_id(entry["header"])
            if not uniprot_id or uniprot_id in seen_ids:
                continue
            seen_ids.add(uniprot_id)
            candidates.append({
                "uniprot_id": uniprot_id,
                "identity": hit["identity"],
                "evalue": hit["evalue"],
                "tstart": hit["tstart"],
                "tend": hit["tend"],
                "tlen": hit["tlen"],
                "aligned_seq": entry["sequence"],
            })

        # Sort by identity (best first), take top N
        candidates.sort(key=lambda c: c["identity"], reverse=True)
        candidates = candidates[:max_candidates]

        if not candidates:
            return None

        # Evaluate each candidate
        evaluated = []  # list of (candidate_dict, plddt)
        for cand in candidates:
            cif_path = self.download_afdb_structure(cand["uniprot_id"], cache_dir)
            if cif_path is None:
                continue

            # Parse pLDDT for aligned region (1-based)
            tstart = max(1, cand["tstart"])
            tend = cand["tend"]
            mean_plddt = self.parse_plddt_for_region(cif_path, tstart, tend)

            if mean_plddt < plddt_threshold:
                continue

            cand["cif_path"] = cif_path
            cand["plddt"] = mean_plddt
            evaluated.append((cand, mean_plddt))

            # Tier 1: perfect identity + high pLDDT -> instant accept
            if cand["identity"] >= 1.0 and mean_plddt >= perfect_plddt:
                return cand

        if not evaluated:
            return None

        # Tier 2: >95% identity, highest pLDDT
        tier2 = [(c, p) for c, p in evaluated if c["identity"] > 0.95]
        if tier2:
            return max(tier2, key=lambda x: x[1])[0]

        # Tier 3: best overall pLDDT
        return max(evaluated, key=lambda x: x[1])[0]

    def write_template_a3m(self, query_seq, template_hit, output_path):
        """Write Protenix-compatible template .a3m file.

        Protenix parses headers with regex:
            ^>?([a-z0-9]+)_(\\w+)/([0-9]+)-([0-9]+).*mol:protein.*protein length:([0-9]+)

        Header format: >{pdb_id}_{chain}/{start}-{end} mol:protein length:{total}
        """
        uniprot_id = template_hit["uniprot_id"]
        # pdb_id must be lowercase alphanumeric only
        pdb_id = re.sub(r"[^a-z0-9]", "", f"af{uniprot_id}".lower())
        chain = "A"
        start = max(1, template_hit["tstart"])
        end = template_hit["tend"]
        total = template_hit.get("tlen", end + 50)

        header = f">{pdb_id}_{chain}/{start}-{end} mol:protein length:{total}"

        # The aligned sequence from the MSA (a3m format)
        aligned_seq = template_hit.get("aligned_seq", "-" * len(query_seq))

        content = f">query\n{query_seq}\n{header}\n{aligned_seq}\n"
        with open(output_path, "w") as f:
            f.write(content)


    # == PDB Hit Search (RCSB) =========================================

    def search_pdb_hits(self, query_seq, min_identity=0.30, max_hits=5):
        """Search RCSB PDB for structures matching query sequence.

        Uses the RCSB PDB Search API v2 with MMseqs2 backend.
        Returns list of hits sorted by score (best first).
        """
        url = "https://search.rcsb.org/rcsbsearch/v2/query"
        query = {
            "query": {
                "type": "terminal",
                "service": "sequence",
                "parameters": {
                    "evalue_cutoff": 0.1,
                    "identity_cutoff": min_identity,
                    "sequence_type": "protein",
                    "value": query_seq
                }
            },
            "return_type": "polymer_entity",
            "request_options": {
                "scoring_strategy": "sequence",
                "sort": [{"sort_by": "score", "direction": "desc"}],
                "paginate": {"start": 0, "rows": max_hits}
            }
        }

        try:
            resp = requests.post(url, json=query, timeout=60)
            if resp.status_code == 204:
                return []  # No results
            if resp.status_code != 200:
                return []
            data = resp.json()
            hits = []
            for item in data.get("result_set", []):
                identifier = item.get("identifier", "")
                score = item.get("score", 0)
                parts = identifier.split("_")
                if len(parts) >= 2:
                    hits.append({
                        "pdb_id": parts[0].upper(),
                        "entity_id": parts[1],
                        "score": score,
                        "identifier": identifier,
                    })
            return hits
        except Exception as e:
            print(f"    PDB search error: {e}")
            return []

    def download_pdb_structure(self, pdb_id, cache_dir):
        """Download mmCIF structure from RCSB PDB.

        Returns cached path or None on error.
        """
        import urllib.request, urllib.error
        os.makedirs(cache_dir, exist_ok=True)
        cached = os.path.join(cache_dir, f"{pdb_id.upper()}.cif")

        if os.path.exists(cached):
            return cached

        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.cif"
        try:
            urllib.request.urlretrieve(url, cached)
            return cached
        except (urllib.error.HTTPError, urllib.error.URLError) as e:
            print(f"    PDB download failed for {pdb_id}: {e}")
            return None

    def get_pdb_metadata(self, pdb_id):
        """Get resolution, release date, and title from RCSB Data API.

        Returns dict with metadata or empty dict on failure.
        """
        url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code != 200:
                return {}
            data = resp.json()
            result = {
                "title": data.get("struct", {}).get("title", ""),
                "release_date": data.get("rcsb_accession_info", {}).get("initial_release_date", ""),
            }
            resolution = None
            if "rcsb_entry_info" in data:
                resolution = data["rcsb_entry_info"].get("resolution_combined")
                if resolution and isinstance(resolution, list):
                    resolution = resolution[0]
            result["resolution"] = resolution
            return result
        except Exception:
            return {}

    # == Output Helpers ================================================

    def entries_to_a3m(self, entries):
        # Convert list of entry dicts back to a3m format string.
        lines = []
        for entry in entries:
            lines.append(entry["header"])
            lines.append(entry["sequence"])
        return "\n".join(lines) + "\n"

    def truncate_msa(self, entries, max_seqs):
        # Truncate entries to max_seqs depth (including query row at index 0).
        if max_seqs <= 0 or len(entries) <= max_seqs:
            return entries
        return entries[:max_seqs]


# == Instantiate global processor ======================================
processor = MSAProcessor()
print("\u2713 MSAProcessor class loaded successfully.")
print(f"  Methods: process_csv, get_unique_proteins, search_msa, parse_a3m_headers,")
print(f"  extract_best_template, write_template_a3m, search_pdb_hits,")
print(f"  download_pdb_structure, get_pdb_metadata, entries_to_a3m, truncate_msa")




# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
if 'global_settings' not in globals() or 'csv_path' not in global_settings:
    raise ValueError('Please run Cell 1 (Upload CSV) first')

csv_path = global_settings['csv_path']

jobs, summary_df = processor.process_csv(csv_path)

if not jobs:
    raise RuntimeError("No valid jobs found in CSV. Check format.")

print("\n" + "\u2501" * 60)
print(f"\u25b6 Job Summary ({len(jobs)} jobs)")
print("\u2501" * 60)
print(summary_df.to_string(index=False))

unique_proteins = processor.get_unique_proteins(jobs)

total_proteins = sum(
    sum(1 for c in j["chains"] if c["type"] == "protein")
    for j in jobs
)

print("\n" + "\u2501" * 60)
print(f"\u25b6 Unique Proteins: {len(unique_proteins)} (from {total_proteins} total protein chains)")
print("\u2501" * 60)
for filename, seq in unique_proteins.items():
    print(f"  {filename}: {len(seq)} residues, {seq[:30]}{'...' if len(seq) > 30 else ''}")

# Update global_settings with processed data
global_settings["jobs"] = jobs
global_settings["unique_proteins"] = unique_proteins

print("\n" + "\u2501" * 60)
print(f"\u2713 Ready. {len(unique_proteins)} proteins to search, {len(jobs)} jobs to pair.")
print(f"  Output folder: {global_settings['output_folder_name']}")
gdrive_status = "connected" if global_settings["gdrive_enabled"] else "disabled"
print(f"  Google Drive: {gdrive_status}")
print("\u2501" * 60)

In [ ]:
#@title Cell 4: MSA Search and Template Settings
#@markdown Configure MSA search parameters and AFDB template extraction.

# \u2500\u2500 MSA Search Settings \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
#@markdown ### MSA Search
msa_server_url = "https://api.colabfold.com"  #@param {type:"string"}
#@markdown ColabFold API endpoint. Default is the public server.

msa_database = "UniRef+EnvDB"  #@param ["UniRef+EnvDB", "UniRef only"]
#@markdown **UniRef+EnvDB**: Broader search, more homologs (recommended). **UniRef only**: Faster, fewer results.

pair_mode = "unpaired_paired"  #@param ["unpaired_paired", "paired", "unpaired"]
#@markdown **unpaired_paired**: Both individual and paired MSAs (recommended). **paired**: Only cross-chain paired. **unpaired**: Only per-chain.

#@markdown ---
#@markdown ### AFDB Templates & PDB Hits
extract_templates = True  #@param {type:"boolean"}
#@markdown Download AFDB structural templates and best PDB hits (>30% identity)
#@markdown for each protein chain. AFDB templates are selected by identity + pLDDT.
#@markdown PDB hits are the best experimental structures from the Protein Data Bank.
#@markdown **All-or-nothing**: AFDB templates used only if ALL protein chains in a job get one.

#@markdown ---
#@markdown ### MSA Depth
max_msa = "auto"  #@param ["auto", "512:1024", "256:512", "128:256", "64:128"]
#@markdown Maximum MSA depth (paired:unpaired). **auto**: Use full MSA as returned by server. Lower values reduce memory in folding tools.

rate_limit_delay = 5  #@param {type:"slider", min:1, max:30, step:1}
#@markdown Seconds to wait between API requests (rate limiting). ColabFold public server may throttle rapid requests.

# \u2500\u2500 Map settings \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
database_map = {"UniRef+EnvDB": "uniref_envdb", "UniRef only": "uniref"}
db_value = database_map.get(msa_database, "uniref_envdb")

# Parse max_msa
max_seqs_val = MSAProcessor.MAX_SEQS_OPTIONS.get(max_msa, 0)

# \u2500\u2500 Store in global_settings \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
global_settings["config"] = {
    "msa_server_url": msa_server_url,
    "msa_database": db_value,
    "pair_mode": pair_mode,
    "extract_templates": extract_templates,
    "max_msa": max_msa,
    "max_seqs": max_seqs_val,
    "rate_limit_delay": rate_limit_delay,
}

# \u2500\u2500 Print summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print("\u2501" * 60)
print("\u25b6 MSA Configuration Summary")
print("\u2501" * 60)
print(f"  Server URL:          {msa_server_url}")
print(f"  Database:            {msa_database} ({db_value})")
print(f"  Pair mode:           {pair_mode}")
print(f"  Extract templates + PDB hits: {extract_templates}")
print(f"  Max MSA depth:       {max_msa}")
print(f"  Rate limit delay:    {rate_limit_delay}s")
print("\u2501" * 60)


In [ ]:
#@title Cell 5: Run MSA Search, Templates & Download
#@markdown Searches MSAs for each unique protein, extracts AFDB templates (optional),
#@markdown writes query-only paired MSAs, saves results to Google Drive.
%%time

import os, json, time, zipfile
import pandas as pd

# \u2500\u2500 Setup \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
jobs = global_settings["jobs"]
unique_proteins = global_settings["unique_proteins"]
config = global_settings["config"]
drive = global_settings["drive"]
folder_id = global_settings["folder_id"]
gdrive_enabled = global_settings["gdrive_enabled"]
output_folder_name = global_settings.get("output_folder_name", "MSA_Results")

# Overwrite protection: append _2, _3, etc. if output dir already exists with content
output_base = f"/content/{output_folder_name}"
if os.path.exists(output_base) and os.listdir(output_base):
    counter = 2
    while os.path.exists(f"{output_base}_{counter}") and os.listdir(f"{output_base}_{counter}"):
        counter += 1
    output_base = f"{output_base}_{counter}"
    output_folder_name = f"{output_folder_name}_{counter}"
    print(f"  Previous output exists. Using: {output_folder_name}")

individual_dir = os.path.join(output_base, "individual")
paired_dir = os.path.join(output_base, "paired")
templates_base = os.path.join(output_base, "templates")
afdb_cache_dir = os.path.join(output_base, ".cache", "afdb")
pdb_cache_dir = os.path.join(output_base, ".cache", "pdb")
os.makedirs(individual_dir, exist_ok=True)
os.makedirs(paired_dir, exist_ok=True)

server_url = config["msa_server_url"]
database = config["msa_database"]
pair_mode_cfg = config["pair_mode"]
extract_templates = config.get("extract_templates", False)
max_seqs = config["max_seqs"]
rate_limit_delay = config["rate_limit_delay"]

run_start_time = time.time()
search_stats = {"searched": 0, "failed": 0, "cached": 0}
template_stats = {"extracted": 0, "failed": 0, "skipped": 0}

print("\u2501" * 60)
print("\u25b6 Phase 1: MSA Search")
print(f"  {len(unique_proteins)} unique proteins to search")
print(f"  Output: {output_base}")
print("\u2501" * 60)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 1: MSA Search (sequential with rate limiting)
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
msa_cache = {}

for idx, (filename, sequence) in enumerate(unique_proteins.items()):
    a3m_path = os.path.join(individual_dir, f"{filename}.a3m")

    # Check if already exists (from previous run)
    if os.path.exists(a3m_path):
        print(f"\n[{idx+1}/{len(unique_proteins)}] {filename} ({len(sequence)} res) \u2014 cached")
        with open(a3m_path, "r") as f:
            msa_cache[filename] = f.read()
        search_stats["cached"] += 1
        continue

    print(f"\n[{idx+1}/{len(unique_proteins)}] {filename} ({len(sequence)} res) \u2014 searching...")
    search_start = time.time()

    a3m_content = processor.search_msa(
        sequence,
        server_url=server_url,
        database=database,
        pair_mode=pair_mode_cfg,
    )

    search_elapsed = time.time() - search_start

    if a3m_content:
        # Apply truncation if configured
        if max_seqs and max_seqs != 0:
            entries = processor.parse_a3m_headers(a3m_content)
            if isinstance(max_seqs, tuple):
                max_depth = max_seqs[1]  # use unpaired limit
            else:
                max_depth = int(max_seqs) if isinstance(max_seqs, (int, float)) and max_seqs > 0 else 0
            if max_depth > 0:
                entries = processor.truncate_msa(entries, max_depth)
                a3m_content = processor.entries_to_a3m(entries)

        # Save individual MSA
        with open(a3m_path, "w") as f:
            f.write(a3m_content)
        msa_cache[filename] = a3m_content

        entries = processor.parse_a3m_headers(a3m_content)
        n_homologs = len(entries) - 1  # minus query
        print(f"  \u2713 {n_homologs} homologs ({search_elapsed:.1f}s)")
        search_stats["searched"] += 1
    else:
        print(f"  \u2717 MSA search failed ({search_elapsed:.1f}s)")
        search_stats["failed"] += 1

    # Rate limiting between requests
    if idx < len(unique_proteins) - 1:
        time.sleep(rate_limit_delay)

# Phase 1 summary
print("\n" + "\u2501" * 60)
phase1_elapsed = time.time() - run_start_time
print(f"\u2713 Phase 1 complete ({phase1_elapsed:.1f}s)")
print(f"  Searched: {search_stats['searched']}, Cached: {search_stats['cached']}, Failed: {search_stats['failed']}")
print("\u2501" * 60)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 1.5: AFDB Template Extraction (optional)
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
template_results = {}  # filename -> template_hit dict or None

if extract_templates:
    os.makedirs(afdb_cache_dir, exist_ok=True)

    print(f"\n\u25b6 Phase 1.5: AFDB Template Extraction")
    print(f"  Checking {len(unique_proteins)} proteins for AFDB templates")
    print("\u2501" * 60)

    phase15_start = time.time()

    for idx, (filename, sequence) in enumerate(unique_proteins.items()):
        print(f"\n[{idx+1}/{len(unique_proteins)}] {filename}")

        if filename not in msa_cache:
            print(f"  \u2014 No MSA available, skipping templates")
            template_results[filename] = None
            template_stats["skipped"] += 1
            continue

        # Per-chain AFDB output dir
        chain_afdb_dir = os.path.join(templates_base, filename, "AFDB")
        os.makedirs(chain_afdb_dir, exist_ok=True)

        a3m_content = msa_cache[filename]
        tmpl_a3m_path = os.path.join(chain_afdb_dir, f"{filename}_templates.a3m")

        # Check if already extracted
        if os.path.exists(tmpl_a3m_path):
            print(f"  \u2713 Template .a3m already exists (cached)")
            template_results[filename] = {"cached": True, "a3m_path": tmpl_a3m_path}
            template_stats["extracted"] += 1
            continue

        best = processor.extract_best_template(
            a3m_content, sequence, afdb_cache_dir
        )

        if best:
            processor.write_template_a3m(sequence, best, tmpl_a3m_path)
            # Copy AFDB .cif with original name (UniProt ID)
            import shutil as _shutil
            if best.get("cif_path") and os.path.isfile(best["cif_path"]):
                _dest_cif = os.path.join(chain_afdb_dir, f"{best['uniprot_id']}.cif")
                if not os.path.exists(_dest_cif):
                    _shutil.copy2(best["cif_path"], _dest_cif)
            template_results[filename] = best
            template_stats["extracted"] += 1
            print(f"  \u2713 AFDB: {best['uniprot_id']} "
                  f"(identity={best['identity']:.0%}, pLDDT={best['plddt']:.1f})")
        else:
            template_results[filename] = None
            template_stats["failed"] += 1
            print(f"  \u2717 No suitable AFDB template found")

    phase15_elapsed = time.time() - phase15_start
    print("\n" + "\u2501" * 60)
    print(f"\u2713 Phase 1.5 complete ({phase15_elapsed:.1f}s)")
    print(f"  Templates found: {template_stats['extracted']}, "
          f"Not found: {template_stats['failed']}, "
          f"Skipped: {template_stats['skipped']}")
    print("\u2501" * 60)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 1.6: PDB Hit Search (>30% identity)
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
pdb_results = {}  # filename -> list of hit dicts or None
pdb_stats = {"found": 0, "failed": 0, "skipped": 0}

if extract_templates:
    os.makedirs(pdb_cache_dir, exist_ok=True)

    print(f"\n\u25b6 Phase 1.6: PDB Hit Search (>30% identity)")
    print(f"  Searching RCSB PDB for {len(unique_proteins)} proteins")
    print("\u2501" * 60)

    phase16_start = time.time()

    for idx, (filename, sequence) in enumerate(unique_proteins.items()):
        print(f"\n[{idx+1}/{len(unique_proteins)}] {filename}")

        # Per-chain PDB output dir
        chain_pdb_dir = os.path.join(templates_base, filename, "PDB")
        os.makedirs(chain_pdb_dir, exist_ok=True)

        # Check if already have PDB hits (from previous run)
        existing_cifs = [f for f in os.listdir(chain_pdb_dir) if f.endswith('.cif')]
        if existing_cifs:
            print(f"  \u2713 {len(existing_cifs)} PDB hit(s) already downloaded (cached)")
            pdb_results[filename] = [{"cached": True, "cif": c} for c in existing_cifs]
            pdb_stats["found"] += 1
            continue

        # Search RCSB PDB (top 5 hits)
        hits = processor.search_pdb_hits(sequence, min_identity=0.30, max_hits=5)
        if not hits:
            print(f"  \u2014 No PDB hits >30% identity")
            pdb_results[filename] = None
            pdb_stats["failed"] += 1
            continue

        hit_records = []
        for hit in hits:
            pdb_id = hit["pdb_id"]
            score = hit["score"]

            # Download structure to shared cache
            cif_path = processor.download_pdb_structure(pdb_id, pdb_cache_dir)
            if cif_path is None:
                continue

            # Copy to per-chain PDB dir with original PDB name
            import shutil as _shutil_pdb
            dest_path = os.path.join(chain_pdb_dir, f"{pdb_id}.cif")
            if not os.path.exists(dest_path):
                _shutil_pdb.copy2(cif_path, dest_path)

            # Get metadata
            meta = processor.get_pdb_metadata(pdb_id)

            hit_record = {
                "pdb_id": pdb_id,
                "entity_id": hit["entity_id"],
                "score": score,
                "resolution": meta.get("resolution"),
                "title": meta.get("title", "")[:80],
                "release_date": meta.get("release_date", ""),
            }
            hit_records.append(hit_record)

            res_val = meta.get("resolution")
            res_str = f"{res_val:.2f}\u00c5" if res_val else "N/A"
            print(f"  \u2713 PDB: {pdb_id} (score={score:.2f}, resolution={res_str})")

        if hit_records:
            pdb_results[filename] = hit_records
            pdb_stats["found"] += 1
            print(f"  {len(hit_records)} PDB structure(s) downloaded")
        else:
            pdb_results[filename] = None
            pdb_stats["failed"] += 1
            print(f"  \u2717 All PDB downloads failed")

        # Brief delay between proteins to avoid hammering RCSB
        time.sleep(1)

    phase16_elapsed = time.time() - phase16_start
    print("\n" + "\u2501" * 60)
    print(f"\u2713 Phase 1.6 complete ({phase16_elapsed:.1f}s)")
    print(f"  Proteins with PDB hits: {pdb_stats['found']}, "
          f"No hits: {pdb_stats['failed']}")
    print("\u2501" * 60)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 2: Write query-only paired MSAs + template decisions
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
print(f"\n\u25b6 Phase 2: Write Paired MSAs (query-only)")
print(f"  {len(jobs)} jobs to process")
print("\u2501" * 60)

phase2_start = time.time()
template_decisions = {}  # jobname -> {use_templates, reason, chains}

for job_idx, job in enumerate(jobs):
    jobname = job["jobname"]
    protein_chains = [c for c in job["chains"] if c["type"] == "protein"]

    print(f"\n[{job_idx+1}/{len(jobs)}] {jobname}: {len(protein_chains)} protein chain(s)")

    if not protein_chains:
        print(f"  \u2014 No protein chains, skipping")
        continue

    # Write query-only paired MSA for each protein chain
    chains_with_templates = []
    chains_without_templates = []

    for chain in protein_chains:
        msa_filename = chain.get("_msa_filename")
        chain_label = chain["name"] if chain["name"] else msa_filename

        if msa_filename and msa_filename in msa_cache:
            # Write query-only: just first entry (header + sequence)
            entries = processor.parse_a3m_headers(msa_cache[msa_filename])
            if entries:
                query_entry = entries[0]
                paired_path = os.path.join(paired_dir, f"{jobname}_{chain_label}_paired.a3m")
                with open(paired_path, "w") as f:
                    f.write(f"{query_entry['header']}\n{query_entry['sequence']}\n")
                print(f"  \u2713 {chain_label} -> paired/ (query-only)")
            else:
                print(f"  \u2717 {chain_label}: empty MSA")
        else:
            print(f"  \u2717 {chain_label}: no MSA available")

        # Track template availability
        if extract_templates and msa_filename:
            tmpl = template_results.get(msa_filename)
            if tmpl is not None:
                chains_with_templates.append(chain_label)
            else:
                chains_without_templates.append(chain_label)

    # All-or-nothing template decision per job
    if extract_templates and protein_chains:
        if chains_without_templates:
            template_decisions[jobname] = {
                "use_templates": False,
                "reason": f"missing templates for: {', '.join(chains_without_templates)}",
                "chains_with": chains_with_templates,
                "chains_without": chains_without_templates,
            }
            print(f"  Templates: DISABLED (missing: {', '.join(chains_without_templates)})")
        elif chains_with_templates:
            template_decisions[jobname] = {
                "use_templates": True,
                "reason": f"all {len(chains_with_templates)} chain(s) have templates",
                "chains_with": chains_with_templates,
                "chains_without": [],
            }
            print(f"  Templates: ENABLED ({len(chains_with_templates)} chain(s))")
        else:
            template_decisions[jobname] = {
                "use_templates": False,
                "reason": "no protein chains with MSA",
                "chains_with": [],
                "chains_without": [],
            }

phase2_elapsed = time.time() - phase2_start
print("\n" + "\u2501" * 60)
print(f"\u2713 Phase 2 complete ({phase2_elapsed:.1f}s)")
if extract_templates:
    enabled = sum(1 for d in template_decisions.values() if d["use_templates"])
    disabled = sum(1 for d in template_decisions.values() if not d["use_templates"])
    print(f"  Template decisions: {enabled} enabled, {disabled} disabled")
print("\u2501" * 60)

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# PHASE 3: Write metadata and upload to GDrive
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
total_elapsed = time.time() - run_start_time

# Build per-protein template summary for metadata (JSON-serializable)
template_summary = {}
if extract_templates:
    for filename, tmpl in template_results.items():
        if tmpl is None:
            template_summary[filename] = {"found": False}
        elif tmpl.get("cached"):
            template_summary[filename] = {"found": True, "cached": True}
        else:
            template_summary[filename] = {
                "found": True,
                "uniprot_id": tmpl.get("uniprot_id", ""),
                "identity": tmpl.get("identity", 0),
                "plddt": tmpl.get("plddt", 0),
            }

# Build PDB results summary for metadata (JSON-serializable)
pdb_summary = {}
if extract_templates:
    for filename, pdb_hit_list in pdb_results.items():
        if pdb_hit_list is None:
            pdb_summary[filename] = {"found": False}
        elif isinstance(pdb_hit_list, list) and pdb_hit_list and pdb_hit_list[0].get("cached"):
            pdb_summary[filename] = {"found": True, "cached": True, "count": len(pdb_hit_list)}
        elif isinstance(pdb_hit_list, list):
            pdb_summary[filename] = {
                "found": True,
                "count": len(pdb_hit_list),
                "hits": [{k: v for k, v in h.items() if k != "cif_path"} for h in pdb_hit_list],
            }
        else:
            pdb_summary[filename] = {"found": False}

# Write metadata.json
metadata = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "output_folder": output_folder_name,
    "config": config,
    "search_stats": search_stats,
    "template_stats": template_stats if extract_templates else {},
    "template_results": template_summary,
    "template_decisions": template_decisions,
    "pdb_stats": pdb_stats if extract_templates else {},
    "pdb_results": pdb_summary,
    "total_elapsed_seconds": round(total_elapsed, 1),
    "unique_proteins": {k: len(v) for k, v in unique_proteins.items()},
}
metadata_path = os.path.join(output_base, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\n\u2713 Wrote metadata.json")

# Zip everything (use dynamic name matching output folder)
zip_path = f"/content/{output_folder_name}.zip"
print(f"\u25b6 Creating ZIP archive...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, zip_files in os.walk(output_base):
        for zf in zip_files:
            file_path = os.path.join(root, zf)
            arcname = os.path.relpath(file_path, "/content")
            zipf.write(file_path, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"  \u2713 {output_folder_name}.zip ({zip_size_mb:.1f} MB)")

# Upload to GDrive
gdrive_link = None
if gdrive_enabled and drive and folder_id:
    print(f"\n\u25b6 Uploading to Google Drive...")
    try:
        def upload_to_gdrive(drv, file_path, fid):
            uploaded_file = drv.CreateFile({
                "title": os.path.basename(file_path),
                "parents": [{"id": fid}]
            })
            uploaded_file.SetContentFile(file_path)
            uploaded_file.Upload()
            return f"https://drive.google.com/file/d/{uploaded_file['id']}/view"

        gdrive_link = upload_to_gdrive(drive, zip_path, folder_id)
        print(f"  \u2713 Uploaded to Google Drive: {gdrive_link}")

        # Also upload individual and paired dirs as separate zips for convenience
        for subdir_name in ["individual", "paired", "templates"]:
            subdir_path = os.path.join(output_base, subdir_name)
            if os.path.isdir(subdir_path) and os.listdir(subdir_path):
                sub_zip = f"/content/{output_folder_name}_{subdir_name}.zip"
                with zipfile.ZipFile(sub_zip, "w", zipfile.ZIP_DEFLATED) as szf:
                    for root, dirs, sub_files in os.walk(subdir_path):
                        for sf in sub_files:
                            fp = os.path.join(root, sf)
                            arcname = os.path.relpath(fp, subdir_path)
                            szf.write(fp, arcname)
                sub_link = upload_to_gdrive(drive, sub_zip, folder_id)
                print(f"  \u2713 Uploaded {output_folder_name}_{subdir_name}.zip: {sub_link}")
    except Exception as e:
        print(f"  \u2717 GDrive upload failed: {e}")
        print(f"    Results still available locally at {zip_path}")

# Local download fallback
if not gdrive_enabled or not gdrive_link:
    print(f"\n\u25b6 Downloading results locally...")
    try:
        from google.colab import files as colab_files
        colab_files.download(zip_path)
    except Exception as e:
        print(f"  Results saved at: {zip_path}")

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# FINAL SUMMARY
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
print(f"\n{'=' * 60}")
print(f" MSA Colab \u2014 Run Complete")
print(f"{'=' * 60}")
print(f"  Total time:          {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"  Proteins searched:   {search_stats['searched']}")
print(f"  From cache:          {search_stats['cached']}")
print(f"  Search failures:     {search_stats['failed']}")
if extract_templates:
    print(f"  AFDB templates found:  {template_stats['extracted']}")
    print(f"  AFDB templates failed: {template_stats['failed']}")
    enabled = sum(1 for d in template_decisions.values() if d["use_templates"])
    print(f"  Jobs with templates:   {enabled}/{len(template_decisions)}")
    print(f"  PDB hits:              {pdb_stats['found']} proteins with hits, {pdb_stats['failed']} without")
print(f"  Output: {output_base}/")

# List output files
individual_files = os.listdir(individual_dir) if os.path.isdir(individual_dir) else []
paired_files = os.listdir(paired_dir) if os.path.isdir(paired_dir) else []
template_chain_dirs = os.listdir(templates_base) if os.path.isdir(templates_base) else []
template_chain_dirs = [d for d in template_chain_dirs if not d.startswith('.') and os.path.isdir(os.path.join(templates_base, d))]
print(f"    individual/: {len(individual_files)} files")
print(f"    paired/:     {len(paired_files)} files")
if extract_templates and template_chain_dirs:
    print(f"    templates/:  {len(template_chain_dirs)} chain(s)")
    for _tcd in sorted(template_chain_dirs):
        _afdb_d = os.path.join(templates_base, _tcd, "AFDB")
        _pdb_d = os.path.join(templates_base, _tcd, "PDB")
        _afdb_n = len([f for f in os.listdir(_afdb_d) if f.endswith('.cif')]) if os.path.isdir(_afdb_d) else 0
        _pdb_n = len([f for f in os.listdir(_pdb_d) if f.endswith('.cif')]) if os.path.isdir(_pdb_d) else 0
        print(f"      {_tcd}/  AFDB: {_afdb_n} cif  PDB: {_pdb_n} cif")

if gdrive_link:
    print(f"  Google Drive:  {gdrive_link}")
print(f"\n  To use with folding notebooks, set msa_folder to:")
print(f"    {output_folder_name}")
print(f"{'=' * 60}")
